In [ ]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-05-18 18:17:42--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.5.0/cloudflared-linux-amd64 [following]
--2026-05-18 18:17:42--  https://github.com/cloudflare/cloudflared/releases/download/2026.5.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/19374280-4acc-49fd-a5af-eb955320fe42?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-18T18%3A59%3A58Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-18T1

In [ ]:
!pip install streamlit-option-menu

In [20]:
%%writefile Diamond.py

import streamlit as st
from streamlit_option_menu import option_menu
import pandas as pd
import pickle

# ---------------------- CONFIG ----------------------

st.set_page_config(page_title="Diamond Dynamics", page_icon="💎", layout="wide")
st.title("💎 Diamond Dynamics")

# ---------------------- SIDEBAR ----------------------
with st.sidebar:
    page = option_menu(
        menu_title="🌐 Navigation",
        options=["📊 Price Prediction", "🏆 Market Segmentation"],
        icons=[" ", " ", " "],
        menu_icon =" ",
        default_index=0,
    )

# ======================================================
# PAGE 1 : PRICE PREDICTION
# ======================================================
if page == "📊 Price Prediction":
    # --- LOAD MODEL ---
    model_path = '/content/drive/MyDrive/datascience/Mini Projects/Diamond Dynamics: Price Prediction and Market Segmentation/Diamond_price.pkl'

    @st.cache_resource
    def load_assets():
        try:
            with open(model_path, "rb") as f:
                return pickle.load(f)
        except Exception as e:
            st.error(f"Error loading model: {e}")
            return None

    model = load_assets()

    # --- INPUTS ---
    st.subheader("💎 Diamond Attribute Profiler")
    col1, col2 = st.columns(2)

    with col1:
        carat = st.number_input("Carat Weight", min_value=0.2, value=1.03)
        x = st.number_input("Length (x) in mm", min_value=1.96, value=6.55)
        y = st.number_input("Width (y) in mm", min_value=1.99, value=6.44)
        z = st.number_input("Depth (z) in mm", min_value=1.23, value=4.03)
        table = st.number_input("Table %", min_value=51.5, value=56.0)

    with col2:
        cut = st.selectbox("Cut Quality", ["Ideal", "Premium", "Very Good", "Good", "Fair"])
        color = st.selectbox("Color Grade", ["D", "E", "F", "G", "H", "I", "J"])
        clarity = st.selectbox("Clarity Grade", ["IF", "VVS1", "VVS2", "VS1", "VS2", "SI1", "SI2", "I1"])
        depth = st.number_input("Depth %", min_value=58.75, value=62.0)


    input_data = pd.DataFrame([{
            'carat': carat,
            'x': x,
            'y': y,
            'z': z,
            'cut': cut,
            'color': color,
            'clarity': clarity,
            'table' : table,
            'depth' : depth
           }])

    # --- PREDICTION ---
    if st.button("🎯 Predict Price", type="primary"):
        if model:
            # Pipeline automatically encodes the labels here
            prediction = model.predict(input_data)[0]
            st.write(prediction)
            # Result in INR
            price_inr = prediction * 95.21
            st.metric("Predicted Price (INR)", f"₹{price_inr:,.2f}")
        else:
            st.error("Model is not available.")

# ======================================================
# PAGE 2 : MARKET SEGMENTATION
# ======================================================

elif page == "🏆 Market Segmentation":
    # --- LOAD MODEL ---
    model_path = '/content/drive/MyDrive/datascience/Mini Projects/Diamond Dynamics: Price Prediction and Market Segmentation/Diamond_cluster.pkl'

    @st.cache_resource
    def load_assets():
        with open(model_path, "rb") as f:
            model = pickle.load(f)
            return model

    try:
        pipeline = load_assets()
    except FileNotFoundError:
        st.error("Error: 'diamond_cluster_model.pkl' file not found. Please train and export the model first.")
        st.stop()

    model = load_assets()

    # --- INPUTS ---
    st.subheader("💎 Diamond Attribute Profiler")
    col1, col2 = st.columns(2)

    with col1:
        carat = st.number_input("Carat Weight", min_value=0.2, value=1.03)
        x = st.number_input("Length (x) in mm", min_value=1.96, value=6.55)
        y = st.number_input("Width (y) in mm", min_value=1.99, value=6.44)
        z = st.number_input("Depth (z) in mm", min_value=1.23, value=4.03)
        table = st.number_input("Table %", min_value=51.5, value=56.0)

    with col2:
        cut = st.selectbox("Cut Quality", ["Ideal", "Premium", "Very Good", "Good", "Fair"])
        color = st.selectbox("Color Grade", ["D", "E", "F", "G", "H", "I", "J"])
        clarity = st.selectbox("Clarity Grade", ["IF", "VVS1", "VVS2", "VS1", "VS2", "SI1", "SI2", "I1"])
        price = st.number_input("Estimated Price (INR)", min_value=31038.46, value=1132083.99)
        depth = st.number_input("Depth %", min_value=58.75, value=62.0)

    # Map cluster IDs to human-readable names
    CLUSTER_NAMES = {
        0: "Affordable Small Diamonds",
        1: "Premium Heavy Diamonds"
    }

    # --- PREDICTION ---
    if st.button("🎯 Predict Cluster Segment", type="primary"):
        # Construct standard dataframe row matches for pipeline ingestion
        input_data = pd.DataFrame([{
            'carat': carat,
            'x': x,
            'y': y,
            'z': z,
            'price': price,
            'cut': cut,
            'color': color,
            'clarity': clarity,
            'table' : table,
            'depth' : depth
           }])

    try:
        cluster_id = pipeline.predict(input_data)[0]
        cluster_name = CLUSTER_NAMES.get(cluster_id, "Unknown Segment Profile")

        # Display Prediction Result Card
        st.success("Analysis Complete!")
        st.metric(label="Assigned Segment Category", value=cluster_name)
        st.caption(f"Internal System Cluster ID Allocation: Cluster {cluster_id}")

    except Exception as e:
        st.error(f"Prediction failed. Feature mismatched structural formats: {e}")

Overwriting Diamond.py


In [ ]:
!streamlit run /content/Diamond.py &>/content/logs.txt &

In [ ]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://comfortable-wines-talked-namely.trycloudflare.com
